# 01 · Define & Explore — the novelty↔foldability problem + the toolkit

**Standard slot:** *define & explore.* **For Project 03 this means:** pin down what
self-consistency (scRMSD) and novelty (TM-score) each measure, stand up `rfdiff_tools`, and run the
"hello-world" — generate 10 monomer backbones, measure foldability + novelty, visualize one (D0).

Run `00_setup.ipynb` first in this session.

## The problem in one figure-of-merit pair

Generative models can invent **novel** folds, but novelty trades off against **foldability**. Two
numbers carry the whole project:

| Metric | Range | Means | Does **not** mean |
|--------|-------|-------|-------------------|
| scRMSD | Å | designed-vs-refolded Cα-RMSD (self-consistency / **foldability**); **< 2 Å** = foldable | binding / function / stability |
| pLDDT | 0–100 | local confidence of the refold | thermostability / ΔG |
| TM-score to PDB | 0–1 | structural similarity to nearest natural fold; **< 0.5 ≈ novel** | a pass/fail of correctness — **novelty is a coordinate, not a verdict** |

The deliverable is the **frontier**: how low can TM-score go (more novel) while scRMSD stays < 2 Å
(still foldable)? Write your own one-paragraph definitions in `D0`, including the "does not mean"
column — that is where most published mistakes live.

## Setup paths

In [ ]:
import sys, os
# Make the project's scripts/ and the cohort's shared/ importable.
# Adjust these if your Colab working directory differs (see 00_setup §5 for Drive mounting).
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

## The backbone-design toolkit

`scripts/rfdiff_tools.py` exposes three functions the campaign uses:
`generate_backbones(length, ss_bias, n, tool)`, `self_consistency(backbone, seq, tool)`, and
`novelty_tm(pdb, tool)`. The real backends (RFdiffusion via ColabDesign, ProteinMPNN, AF2/ESMFold,
Foldseek) need a GPU + heavy installs; a deterministic **mock** backend lets you build and test the
plumbing first. **Never report mock numbers as real designs** — every mock record is flagged
`synthetic=True` and ids are prefixed `EXAMPLE_DATA_`.

In [ ]:
import rfdiff_tools as rt

print("campaign grid:")
print("  lengths   :", rt.CAMPAIGN_LENGTHS)
print("  ss_biases :", rt.SS_BIASES)
print("  foldable  : scRMSD <", rt.SELF_CONSISTENT_SCRMSD, "A")
print("  novel     : TM-score <", rt.NOVEL_TM, "to nearest natural fold")

## Hello-world: generate 10 backbones, score foldability + novelty

Start with the `mock` backend to confirm the plumbing, **then switch `tool="rfdiffusion"`** (and
`"esmfold"`/`"af2"`, `"foldseek"`) on a GPU runtime. The mock numbers encode the project's realistic
priors (short all-α folds best; novelty/length make it harder) so the example looks qualitatively
right — they are **synthetic**, not designs.

In [ ]:
import pandas as pd

backbones = rt.generate_backbones(length=80, ss_bias="alpha", n=10, tool="mock", seed=0)

rows = []
for bb in backbones:
    sc = rt.self_consistency(bb, tool="mock")     # ProteinMPNN(8 seqs)->AF2/ESMFold->best scRMSD
    nv = rt.novelty_tm(bb.backbone_id, tool="mock")  # Foldseek/TM-align vs the PDB
    rows.append(dict(backbone_id=bb.backbone_id, length=bb.length, ss_bias=bb.ss_bias,
                     scrmsd=sc.scrmsd, plddt=sc.plddt, tm_to_pdb=nv.tm_to_pdb,
                     synthetic=bb.synthetic))
demo = pd.DataFrame(rows)
demo.to_csv("results/hello_world.csv", index=False)
print("EXAMPLE_DATA (synthetic) — 10 mock backbones:")
print(demo[["backbone_id", "scrmsd", "plddt", "tm_to_pdb"]].to_string(index=False))
print("\nfoldable (scRMSD<2):", int((demo.scrmsd < rt.SELF_CONSISTENT_SCRMSD).sum()), "/ 10",
      "  novel (TM<0.5):", int((demo.tm_to_pdb < rt.NOVEL_TM).sum()), "/ 10")

### Switch to the real backends

Once `00_setup`'s heavy-install helpers have run this session (RFdiffusion via the ColabDesign
notebook; ProteinMPNN; ESMFold/AF2; Foldseek), change `tool="mock"` to the real backends and rerun.
ESMFold is the fastest refold (no MSA), so use it for triage and AF2 for top picks. **Compute
honesty:** a T4 handles a small batch like this; the *full* campaign in notebook 02 needs an
A100/HPC. Record runtime + the GPU + the pinned commits in `LOG.md`.

In [ ]:
# Uncomment after installing the real backends in 00_setup (GPU runtime):
# real = rt.generate_backbones(length=80, ss_bias="alpha", n=10, tool="rfdiffusion", seed=0)
# sc = rt.self_consistency(real[0], tool="esmfold")   # ProteinMPNN -> ESMFold -> scRMSD
# nv = rt.novelty_tm(real[0].pdb_path, tool="foldseek", db="pdb_db")
# print(sc, nv)
print("Ready — flip tool='mock' to the real backends on a GPU runtime (A100/HPC for the full sweep).")

## Visualize a backbone (py3Dmol)

Use this to eyeball any generated/predicted backbone PDB once you have one. (The mock backend writes
no PDB; this is for real runs.)

In [ ]:
import py3Dmol

def show_pdb(pdb_path_or_str, is_path=True):
    data = open(pdb_path_or_str).read() if is_path else pdb_path_or_str
    view = py3Dmol.view(width=500, height=400)
    view.addModel(data, "pdb")
    view.setStyle({"cartoon": {"color": "spectrum"}})
    view.zoomTo()
    return view.show()

# Example (after a real RFdiffusion run):
# show_pdb(real[0].pdb_path)
print("show_pdb(pdb_path) ready — use it on a real backbone PDB.")

## D0 checklist
- [ ] One-paragraph definition of **scRMSD** and **TM-score/novelty**, each **with** its "does not mean" note (novelty ≠ correctness).
- [ ] Reproduced the 10-backbone hello-world (mock here; one real RFdiffusion→MPNN→refold→Foldseek pass on a GPU runtime).
- [ ] Problem statement with measurable criteria (scRMSD < 2 Å foldable; TM < 0.5 novel) + the controls you'll need.
- [ ] `LOG.md` entry: tool versions/commits, GPU, runtime, seed.

**Next:** `02_generate.ipynb` — run the campaign across lengths × secondary-structure biases.